# Reading checks

In [ ]:
from itertools import islice
from pprint import pprint
import pandas as pd
from datasets import load_dataset

In [2]:
DATASET_ID = "microsoft/OpenMementos"

ds_stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
)

ds_stream

IterableDataset({
    features: ['problem', 'response', 'domain', 'source', 'difficulty'],
    num_shards: 20
})

In [3]:
sample = list(islice(ds_stream, 5))
len(sample)

5

In [4]:
for i, row in enumerate(sample):
    print(f"\n--- row {i} ---")
    print(row.keys())


--- row 0 ---
dict_keys(['problem', 'response', 'domain', 'source', 'difficulty'])

--- row 1 ---
dict_keys(['problem', 'response', 'domain', 'source', 'difficulty'])

--- row 2 ---
dict_keys(['problem', 'response', 'domain', 'source', 'difficulty'])

--- row 3 ---
dict_keys(['problem', 'response', 'domain', 'source', 'difficulty'])

--- row 4 ---
dict_keys(['problem', 'response', 'domain', 'source', 'difficulty'])


In [5]:
pprint(sample[0], width=120, sort_dicts=False)

{'problem': 'The problem over here introduces an new type of strings: if you split the string into equal halfs and '
            'swap the pairs, it produces the same output as sorting the string. We call that a half-sort.\n'
            'Given a purely ASCII string, check if the string is in a half-sort.\n'
            'An example of a half-sort string\n'
            'The string node is a half-sort string, because if you sort by codepoints (note that the codepoints are in '
            'decimal, not binary):\n'
            'n 110\n'
            'o 111\n'
            'd 100\n'
            'e 101\n'
            '\n'
            'That gets turned into:\n'
            'd 100\n'
            'e 101\n'
            'n 110\n'
            'o 111\n'
            '\n'
            "You'll see that the node to deno conversion is exactly moving the right half to the left position.\n"
            'Specification\n'
            '\n'
            'For odd-length strings, splitting should make the first ha

In [6]:
df_sample = pd.DataFrame(sample)
df_sample.head()

,problem,response,domain,source,difficulty
0,The problem over here introduces an new type o...,"<think><|block_start|>Okay, so I need to figur...",code,stackexchange_codegolf,7
1,"Choose a challenge from the Cops thread, deter...","<think><|block_start|>Okay, let's see. I need ...",code,stackexchange_codegolf,7
2,Introduction (may be ignored)\nPutting all pos...,"<think><|block_start|>Okay, I need to solve th...",code,stackexchange_codegolf,8
3,Russian Translation Available\n\nIt is very im...,"<think><|block_start|>Okay, so I have this pro...",code,nvidia/OpenCodeReasoning,8
4,Some built-in functions in PHP return mixed ty...,"<think><|block_start|>Okay, so the user wants ...",code,stackexchange_codegolf,7


In [7]:
column_summary = []

for col in df_sample.columns:
    non_null = [x for x in df_sample[col].tolist() if x is not None]
    example = non_null[0] if non_null else None

    column_summary.append({
        "column": col,
        "python_type": type(example).__name__ if example is not None else None,
        "example_preview": repr(example)[:300],
        "null_count": sum(x is None for x in df_sample[col].tolist()),
    })

pd.DataFrame(column_summary)

,column,python_type,example_preview,null_count
0,problem,str,"""The problem over here introduces an new type ...",0
1,response,str,"'<think><|block_start|>Okay, so I need to figu...",0
2,domain,str,'code',0
3,source,str,'stackexchange_codegolf',0
4,difficulty,int,7,0


In [8]:
for col in df_sample.columns:
    examples = [x for x in df_sample[col].tolist() if x is not None]
    if not examples:
        continue

    x = examples[0]
    if isinstance(x, (list, dict)):
        print(f"\n=== {col} ===")
        print(type(x))
        pprint(x if isinstance(x, dict) else x[:2], width=120, sort_dicts=False)

# Response initial summary statistics

In [9]:
import re

In [10]:
BLOCK_RE = re.compile(r"<\|block_start\|>(.*?)<\|block_end\|>", re.DOTALL)
SUMMARY_RE = re.compile(r"<\|summary_start\|>(.*?)<\|summary_end\|>", re.DOTALL)
THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)

In [11]:
def parse_response(response: str) -> dict:
    if response is None:
        response = ""

    # isolate the <think> section/the reasoning trace of the response
    think_match = THINK_RE.search(response)

    if think_match:
        # isolate the <think> section
        think_text = think_match.group(1)
        # isolate the answer section, removes leading/trailing whitespace
        answer_text = response[think_match.end():].strip()
    else:
        # if no <think> section is present, leave reasoning field empty, the whole response is an answer
        # prevents parsing failures when num blocks == 0
        think_text = ""
        answer_text = response.strip()

    # searches the <think> section for every segment that matches block_re expressions
    # removes leading and trailing whitespace from each extracted block; same for summaries
    # returns two aligned lists, each index in blocks and summaries is the block 
    # and its subsequent summary
    blocks = [x.strip() for x in BLOCK_RE.findall(think_text)]
    summaries = [x.strip() for x in SUMMARY_RE.findall(think_text)]

    return {
        "think_text": think_text,
        "answer_text": answer_text,
        "blocks": blocks,
        "summaries": summaries,
        "n_blocks": len(blocks),
        "n_summaries": len(summaries),
        "think_chars": len(think_text),
        "answer_chars": len(answer_text),
        "response_chars": len(response),
    }

In [12]:
parsed_sample = []

for i, row in enumerate(sample):
    parsed = parse_response(row["response"])

    parsed_sample.append({
        "row": i,
        "domain": row.get("domain"),
        "source": row.get("source"),
        "difficulty": row.get("difficulty"),
        "problem_chars": len(row.get("problem") or ""),
        **{k: v for k, v in parsed.items() if k not in ["think_text", "answer_text", "blocks", "summaries"]},
    })

df_parsed_sample = pd.DataFrame(parsed_sample)
df_parsed_sample

,row,domain,source,difficulty,problem_chars,n_blocks,n_summaries,think_chars,answer_chars,response_chars
0,0,code,stackexchange_codegolf,7,931,7,7,14971,2041,17027
1,1,code,stackexchange_codegolf,7,780,8,8,13694,2065,15774
2,2,code,stackexchange_codegolf,8,2919,4,4,46465,2236,48716
3,3,code,nvidia/OpenCodeReasoning,8,2620,5,5,21086,3760,24861
4,4,code,stackexchange_codegolf,7,604,10,10,20994,3548,24557


In [22]:
df_parsed_sample[[
    "n_blocks",
    "n_summaries",
    "think_chars",
    "answer_chars",
    "response_chars",
]].describe().round(2)

,n_blocks,n_summaries,think_chars,answer_chars,response_chars
count,5.00,5.00,5.00,5.00,5.00
mean,6.80,6.80,23442.00,2730.00,26187.00
std,2.39,2.39,13307.74,850.15,13269.39
min,4.00,4.00,13694.00,2041.00,15774.00
25%,5.00,5.00,14971.00,2065.00,17027.00
50%,7.00,7.00,20994.00,2236.00,24557.00
75%,8.00,8.00,21086.00,3548.00,24861.00
max,10.00,10.00,46465.00,3760.00,48716.00


In [14]:
df_parsed_sample.assign(
    block_summary_delta=df_parsed_sample["n_blocks"] - df_parsed_sample["n_summaries"]
)[[
    "row",
    "n_blocks",
    "n_summaries",
    "block_summary_delta",
]]

,row,n_blocks,n_summaries,block_summary_delta
0,0,7,7,0
1,1,8,8,0
2,2,4,4,0
3,3,5,5,0
4,4,10,10,0


## Second probe on larger portion of the dataset

In [15]:
ds_stream = load_dataset(
    DATASET_ID,
    split="train",
    streaming=True,
)

N_PROBE = 500
probe_rows = []

for i, row in enumerate(islice(ds_stream, N_PROBE)):
    parsed = parse_response(row["response"])

    probe_rows.append({
        "row": i,
        "domain": row.get("domain"),
        "source": row.get("source"),
        "difficulty": row.get("difficulty"),
        "problem_chars": len(row.get("problem") or ""),
        "response_chars": parsed["response_chars"],
        "think_chars": parsed["think_chars"],
        "answer_chars": parsed["answer_chars"],
        "n_blocks": parsed["n_blocks"],
        "n_summaries": parsed["n_summaries"],
        "block_summary_delta": parsed["n_blocks"] - parsed["n_summaries"],
    })

df_probe = pd.DataFrame(probe_rows)
df_probe.head()

,row,domain,source,difficulty,problem_chars,response_chars,think_chars,answer_chars,n_blocks,n_summaries,block_summary_delta
0,0,code,stackexchange_codegolf,7,931,17027,14971,2041,7,7,0
1,1,code,stackexchange_codegolf,7,780,15774,13694,2065,8,8,0
2,2,code,stackexchange_codegolf,8,2919,48716,46465,2236,4,4,0
3,3,code,nvidia/OpenCodeReasoning,8,2620,24861,21086,3760,5,5,0
4,4,code,stackexchange_codegolf,7,604,24557,20994,3548,10,10,0


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/microsoft/OpenMementos/resolve/caaf4bfe9741b8e49253de2d7d07e54567777245/data/train-00000-of-00020.parquet
Retrying in 1s [Retry 1/5].


In [21]:
df_probe[[
    "n_blocks",
    "n_summaries",
    "block_summary_delta",
    "problem_chars",
    "think_chars",
    "answer_chars",
    "response_chars",
]].describe().round(2)

,n_blocks,n_summaries,block_summary_delta,problem_chars,think_chars,answer_chars,response_chars
count,500.00,500.00,500.0,500.00,500.00,500.00,500.00
mean,9.47,9.47,0.0,1987.25,46255.82,2529.26,48800.08
std,2.43,2.43,0.0,1261.77,13546.91,1920.47,13969.62
min,4.00,4.00,0.0,143.00,6182.00,25.00,6728.00
25%,8.00,8.00,0.0,1217.00,37238.25,1228.00,39387.25
50%,9.00,9.00,0.0,1754.00,47547.50,2166.50,49950.50
75%,11.00,11.00,0.0,2482.00,56790.00,3407.50,59873.75
max,21.00,21.00,0.0,11111.00,81093.00,19671.00,82724.00


In [17]:
df_probe.groupby("domain")[["n_blocks", "n_summaries", "think_chars"]].agg(["count", "mean", "median", "max"])

n_blocks                  n_summaries                  think_chars  \
          count  mean median max       count  mean median max       count   
domain                                                                      
code        500  9.47    9.0  21         500  9.47    9.0  21         500   

                                   
             mean   median    max  
domain                             
code    46255.824  47547.5  81093

In [18]:
df_probe.groupby("source")[["n_blocks", "n_summaries", "think_chars"]].agg(["count", "mean", "median", "max"])

n_blocks                      n_summaries            \
                            count      mean median max       count      mean   
source                                                                         
nvidia/OpenCodeReasoning      282  9.535461    9.0  21         282  9.535461   
stackexchange_codegolf        218  9.385321    9.0  18         218  9.385321   

                                    think_chars                                
                         median max       count          mean   median    max  
source                                                                         
nvidia/OpenCodeReasoning    9.0  21         282  46961.879433  48173.0  81093  
stackexchange_codegolf      9.0  18         218  45342.486239  46551.5  71732

In [19]:
df_probe["block_summary_delta"].value_counts().sort_index()

block_summary_delta
0    500
Name: count, dtype: int64

In [20]:
df_probe.query("n_blocks == 0 or n_summaries == 0").head()

,row,domain,source,difficulty,problem_chars,response_chars,think_chars,answer_chars,n_blocks,n_summaries,block_summary_delta


# Initial parsing findings

On the initial streamed probe sample:

- No null values were observed in the top-level fields.
- Each parsed reasoning block has a corresponding summary.
- The block-summary count delta is always zero in the sampled rows.
- The number of block/summary pairs per reasoning trace ranges from 4 to 21.

This supports treating the response as a sequence of aligned `(block_t, summary_t)` pairs followed by a final answer.